# 石油調達→プーリング→製品ブレンドの多期計画

中堅の石油精製・調達会社の「調達計画部」が、数週間の計画期間について、どの原料
(スイート/サワー原油・各種留分)を、いつ、どれだけ買い付け、どの中間タンク(プール)に
入れ、そこからどの製品(プレミアム/レギュラー等)へブレンドするかを決める。

タンクは「よく撹拌された混合物」として扱われ、その硫黄濃度は **硫黄質量在庫 ÷ 体積在庫**
で決まる。払い出し時にはこの濃度で硫黄が出ていくため、**濃度×流量・濃度×在庫の双線形項**が
本質的に現れる — これは近似ではなく、プーリング問題(pooling problem)として古典的に知られる
強い非凸構造そのものである。

各制約の業務的意味は次の通り:

- **調達契約 on/off(固定費 + 最小引取量)**: 供給元と期ごとに契約すると固定費が発生し、
  契約した期は最小引取量以上を買う義務がある(take-or-pay 的)。
- **タンク在庫の期跨ぎ**: 買った原料は中間タンクに貯め、後の期に払い出せる(時間結合)。
- **製品の品質仕様**: 各製品は硫黄分の規格上限を持ち、ブレンドの加重平均濃度がそれを
  超えてはならない。
- **スポット市場バックストップ**: 自社ブレンドで賄えない分は割高なスポット購入で埋める
  (常に実行可能性を担保する現実の逃げ道)。

題材: `samples/manufacturing_and_blending/petroleum_pooling.py`。この notebook では
「素朴な定式化 → 診断 → 診断の中身 → 改善を試す」の順に、この問題が何を難しくしているかを
1本の物語として追う。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/manufacturing_and_blending"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import petroleum_pooling as pp
from pyscipopt import Model, quicksum

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 素朴な定式化

`build_model(scale="default")` を読み込み、規模と非線形項の所在を確認する。既定scaleは
原料8×プール5×製品4×期8。

In [2]:
m0 = pp.build_model("default")
nl = sum(1 for c in m0.getConss() if c.isNonlinear())
print(f"変数 {m0.getNVars()}(うち整数/バイナリ {m0.getNBinVars()})、制約 {m0.getNConss()}"
      f"(うち非線形 {nl})")
print("非線形制約の例(先頭5件、名前で種類がわかる):")
for c in m0.getConss():
    if c.isNonlinear():
        print(" -", c.name)
del m0

変数 696(うち整数/バイナリ 64)、制約 312(うち非線形 112)
非線形制約の例(先頭5件、名前で種類がわかる):
 - sulfur_balance_0_0
 - sulfur_balance_0_1
 - sulfur_balance_0_2
 - sulfur_balance_0_3
 - sulfur_balance_0_4
 - sulfur_balance_0_5
 - sulfur_balance_0_6
 - sulfur_balance_0_7
 - sulfur_balance_1_0
 - sulfur_balance_1_1
 - sulfur_balance_1_2
 - sulfur_balance_1_3
 - sulfur_balance_1_4
 - sulfur_balance_1_5
 - sulfur_balance_1_6
 - sulfur_balance_1_7
 - sulfur_balance_2_0
 - sulfur_balance_2_1
 - sulfur_balance_2_2
 - sulfur_balance_2_3
 - sulfur_balance_2_4
 - sulfur_balance_2_5
 - sulfur_balance_2_6
 - sulfur_balance_2_7
 - sulfur_balance_3_0
 - sulfur_balance_3_1
 - sulfur_balance_3_2
 - sulfur_balance_3_3
 - sulfur_balance_3_4
 - sulfur_balance_3_5
 - sulfur_balance_3_6
 - sulfur_balance_3_7
 - sulfur_balance_4_0
 - sulfur_balance_4_1
 - sulfur_balance_4_2
 - sulfur_balance_4_3
 - sulfur_balance_4_4
 - sulfur_balance_4_5
 - sulfur_balance_4_6
 - sulfur_balance_4_7
 - conc_def_0_0
 - conc_def_0_1
 - conc_def_0_2
 - conc_

`conc_def_*`(濃度の定義: `smass == conc*inv`)と `sulfur_balance_*`(硫黄質量バランス:
`conc*流出量` の項を含む)がプール数×期数だけ現れる。どちらも **濃度(連続)×体積/流量(連続)**
の双線形で、プールの中身が「よく撹拌された混合物」であるという物理そのものから生じる。

## 2. 診断

`mk.analyze` で実際に解き、双対境界の停滞・非線形制約の違反・係数スケールを収集する。

In [3]:
report = mk.analyze(lambda: pp.build_model("default"), name="petroleum-pooling",
                    time_limit=30)
print(report.summary())
print("\ngap:", f"{report.metrics.get('gap', 0):.1%}",
      "/ 支配ボトルネック:", report.metrics.get("bottleneck_type"),
      f"(相対違反 {report.metrics.get('bottleneck_rel_viol', 0):.2f})")

[petroleum-pooling] 検出症状 1件:
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化

gap: 3.8% / 支配ボトルネック: conc_def_2 (相対違反 0.99)


30秒で `numerical_scale`(presolve後も残る係数スケールの桁違い / Big-M候補)が1件発火する。
gapは3.8%程度とさほど大きくない — `conc_def`(濃度定義)の相対違反は大きい(≈1.0、ルートLP緩和解では
ほぼ意味を成さない濃度が出る)ものの、`spatial_share`(空間分枝の双対寄与)は低く、
`weak_relaxation` ルール(空間分枝が支配的なときに発火)は条件を満たさない — SCIPの分枝処理が
非線形の弱さを離散分枝側でかなり吸収できていることを示唆する。次でこの`numerical_scale`の
中身(何がBig-M候補として拾われているか)を直接見る。

## 3. 診断の中身を見る — 静的診断(係数スケール)

`mk.static_diag` 系の関数を直接叩き、presolve後に残る大係数(Big-M候補)を確認する。

In [4]:
from minlpkit.collectors.static_diag import extract_coefficients, scale_summary, residual_scale

m1 = pp.build_model("default")
rs = residual_scale(m1)
print(f"presolve後の係数比 = {rs['ratio']:.3g}(中央値 {rs['median']:.3g})")
bigm_df = None
import pandas as pd
bigm_df = pd.DataFrame(rs["bigm"])
bigm_df

presolve後の係数比 = 2.05e+03(中央値 1)


,name,source,magnitude
0,y_0_0_7,変数境界,709.8
1,y_0_1_7,変数境界,709.8
2,y_0_3_7,変数境界,709.8
3,y_0_2_7,変数境界,709.8
4,y_2_0_7,変数境界,697.4
5,y_2_1_7,変数境界,697.4
6,y_2_3_7,変数境界,697.4
7,y_2_2_7,変数境界,697.4
8,y_1_2_7,変数境界,691.6
9,y_1_3_7,変数境界,691.6


In [5]:
fig = go.Figure(layout=base_layout(
    "presolve後に残る大係数(Big-M候補、中央値比で上位10件)",
    "制約/変数名", "係数の絶対値", height=420))
fig.add_trace(go.Bar(x=bigm_df["name"], y=bigm_df["magnitude"],
    marker_color=[C["warn"] if s == "制約係数" else C["s1"] for s in bigm_df["source"]],
    text=bigm_df["source"], textposition="outside"))
fig.update_xaxes(tickangle=-30)
show(fig)

上位に挙がるのは調達契約の take-or-pay リンク制約(`contract_ub_*`/`contract_lb_*`)まわりで、
`quicksum(x) <= avail[i,t]*z[i,t]` という**Big-M形の連動制約**になっている。ここでの M は
`avail[i,t]`(その期に実際に供給可能な最大量)であり、恣意的に大きい定数ではなく物理的な上限
そのものだが、それでも「制約係数」としては他の項(在庫保管費係数など、オーダー1未満)に比べて
大きく、`residual_bigm_count`側の閾値(中央値の100倍)に引っかかる。

## 4. 改善を試す — Big-M形の連動制約を Indicator制約に置換

`contract_ub`/`contract_lb` は「契約していない(`z=0`)なら購入0、契約している(`z=1`)なら
上限avail・下限min_buyを守る」という論理制約を Big-M で表現したもの。SCIP の
`addConsIndicator` はこの論理をソルバーに直接渡し、M の値を選ぶ必要をなくす
([プレイブック 3. Big-M排除](../../playbook/03-bigm.md)と同じ技法)。`petroleum_pooling.py`
自体には変種を切り替える引数が無いため、この notebook 内で `build_model` と同じデータ生成
(`pp._data`)を使って Indicator 版を組み立てる(既存モデルの `z`/`x` 変数はそのまま再利用し、
Big-M制約2本だけを Indicator制約3本に置き換える)。

In [6]:
def build_indicator(scale="default"):
    '''contract_ub/contract_lb (Big-M) を addConsIndicator に置き換えた変種。'''
    m = pp.build_model(scale)
    z, x = m.data["z"], m.data["x"]
    d = pp._data(scale)
    nF, nL, nT = d["nF"], d["nL"], d["nT"]
    avail, min_buy = d["avail"], d["min_buy"]
    for c in list(m.getConss()):
        if c.name.startswith("contract_ub_") or c.name.startswith("contract_lb_"):
            m.delCons(c)
    for i in range(nF):
        for t in range(nT):
            tot = quicksum(x[i, l, t] for l in range(nL))
            # z=0 なら購入0(Big-Mを使わない厳密な論理制約)
            m.addConsIndicator(tot <= 0, binvar=z[i, t], activeone=False,
                               name=f"ind_off_{i}_{t}")
            # z=1 のときだけ上限avail・下限min_buyを課す
            m.addConsIndicator(tot <= float(avail[i, t]), binvar=z[i, t], activeone=True,
                               name=f"ind_cap_{i}_{t}")
            m.addConsIndicator(-tot <= -float(min_buy[i]), binvar=z[i, t], activeone=True,
                               name=f"ind_lb_{i}_{t}")
    return m

# 等価性の最小確認(small scaleで同じ最適値に到達するか)
mb = pp.build_model("small"); mb.hideOutput(); mb.optimize()
mi = build_indicator("small"); mi.hideOutput(); mi.optimize()
print(f"baseline (Big-M) : obj={mb.getObjVal():.2f}  status={mb.getStatus()}")
print(f"indicator        : obj={mi.getObjVal():.2f}  status={mi.getStatus()}")

baseline (Big-M) : obj=18443.69  status=optimal
indicator        : obj=18443.69  status=optimal


small scaleで同じ最適値に到達する(等価な論理制約の異なる表現)。次に既定scaleで
`mk.compare_variants` により **ルート双対境界・最終gap・ノード数** を比較する。

In [7]:
df = mk.compare_variants(
    {"baseline (Big-M)": lambda: pp.build_model("default"),
     "indicator":        lambda: build_indicator("default")},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,baseline (Big-M),107827.920421,107868.618185,0.038398,222
1,indicator,107675.383008,107747.683035,0.047627,137


In [8]:
base = df.loc[df["variant"] == "baseline (Big-M)"].iloc[0]
ind  = df.loc[df["variant"] == "indicator"].iloc[0]
labels = ["baseline (Big-M)", "indicator"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], ind["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["final_gap"]*100, ind["final_gap"]*100], lambda v: f"{v:.0f}%")
add_bars(3, [base["nodes"], ind["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="Big-M -> Indicator の before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"root_dual : {base['root_dual']:.1f} -> {ind['root_dual']:.1f}")
print(f"final_gap : {base['final_gap']:.1%} -> {ind['final_gap']:.1%}")
print(f"nodes     : {int(base['nodes'])} -> {int(ind['nodes'])}")

root_dual : 107827.9 -> 107675.4
final_gap : 3.8% -> 4.8%
nodes     : 222 -> 137


**正直な検証結果**: このモデルの Big-M(`avail[i,t]`)はそもそも「その期に実際に買える
最大量」という**恣意性のないタイトな値**であり、03番notebook(fixed_charge)の loose Big-M
のような「無駄な緩さ」を持たない。したがって Indicator化による `root_dual`/`final_gap`/`nodes`
の改善は限定的(ほぼ同水準)になりやすい。それでも診断が `numerical_scale` を拾ったのは、
値そのものの妥当性ではなく「他の係数(保管費など)との**桁差**」を見ているためである — つまり
この finding は「モデルの欠陥」ではなく「調達コストと保管費のスケールがそもそも違う実務」を
反映した**自然な数値レンジ**であり、Indicator化してもgapの主因(硫黄濃度の双線形)には触れない。
真にgapへ効かせたいなら、2節で見た`conc_def`/`sulfur_balance`側(濃度×流量の双線形)への
アプローチ(McCormick強化・変数境界タイト化)が本筋になる。

## まとめ

- この問題の本質的な難しさは、**タンクという「貯めて混ぜて後で払い出す」現実の運用**が生む
  濃度×流量・濃度×在庫の双線形(プーリング問題)そのものにある。診断の`numerical_scale`が
  指す Big-M は、実は「調達コストと保管費のスケール差」という業務的に自然な数値レンジであり、
  Indicator化しても劇的には改善しない —— **診断が拾った症状と、gapを支配する真因が別軸に
  あることを正直に確認できた**のがこの notebook の学び。
- 実務的には、この意思決定は「どの原料をいつ買い、どのタンクに貯め、いつどの製品へ払い出すか」
  という調達・在庫・ブレンドの統合計画であり、双線形の強さは近似ではなく物理・化学(混合則)
  そのものに由来する。SCIPは空間分枝限定法でこれを厳密に扱うが、規模が大きくなるほど
  ルート緩和の弱さ(濃度×流量)が効いてくる。

関連: [プレイブック 3. Big-M排除](../../playbook/03-bigm.md) /
[プレイブック 8. 条件数・数値健全性](../../playbook/08-condition.md) /
API [`mk.analyze`](../../api/pipeline.md)